# Measure faithfulness: output drop, ranked removal and locality

Faithfulness metrics ask whether an intervention changes the output the way an explanation claims it should. Three of them on a toy model whose three output classes read disjoint hidden features: output drop, ranked zero-ablation, and locality. The ranked removal is in the spirit of ROAD (Rong, Leemann, Borisov, Kasneci and Kasneci 2022, "A Consistent and Efficient Evaluation Strategy for Attribution Methods", ICML 2022, https://arxiv.org/abs/2202.00449), without ROAD's debiasing imputation step. Before any edit, `tl.validate(scope="intervention")` confirms that a baseline forward of the model replays.

In [ ]:
# recipe-test: tool=torchlens expected=True
from __future__ import annotations

from typing import Any

import torch
from torch import nn

import torchlens as tl


class EvidenceModel(nn.Module):
    """Five hidden features; class 0 reads features 0 and 1, class 1 feature 3, class 2 feature 4."""

    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Linear(5, 5, bias=False)
        self.head = nn.Linear(5, 3, bias=False)
        with torch.no_grad():
            self.features.weight.copy_(torch.eye(5))
            self.head.weight.zero_()
            self.head.weight[0, 0] = 1.0
            self.head.weight[0, 1] = 0.5
            self.head.weight[1, 3] = 1.0
            self.head.weight[2, 4] = 1.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(torch.relu(self.features(x)))


model = EvidenceModel().eval()
x = torch.tensor([[2.0, 1.0, 0.0, 0.2, 0.1]])
report = tl.validate(model, x, scope="intervention")
print("baseline forward replays:", report.is_invariant)  # the one axis this path evaluates
assert report.is_invariant
RESULT = report.is_invariant

In [ ]:
# recipe-test: tool=torchlens expected=0
clean = tl.trace(model, x)
clean_logits = clean.output_ops[-1].out.detach()
target = int(clean_logits.argmax())
print("clean logits", clean_logits, "target class", target)
RESULT = target

Output drop: zero the proposed explanatory site and measure how far the target logit falls. Here the site is the whole ReLU layer.

In [ ]:
# recipe-test: tool=torchlens expected=True
ablated = clean.fork("zero_relu").do(tl.func("relu"), tl.zero_ablate())
ablated_logits = ablated.output_ops[-1].out.detach()
output_drop = float(clean_logits[0, target] - ablated_logits[0, target])
print("output_drop", output_drop)
assert output_drop > 0
RESULT = output_drop > 0

Ranked removal zeros hidden features in order of a ranking and tracks the target logit as more are removed. In real studies the ranking comes from saliency or attribution; here it is each hidden unit's magnitude, so the example stays self-contained.

In [ ]:
# recipe-test: tool=torchlens expected=True
def keep(mask: torch.Tensor):
    """A hook that zeros the hidden features the mask marks 0."""

    def hook(out: torch.Tensor, *, ctx: Any) -> torch.Tensor:
        return out * mask.to(device=out.device, dtype=out.dtype)

    return hook


hidden = clean.find_sites(tl.func("relu")).first().out.detach()[0]
ranking = torch.argsort(hidden.abs(), descending=True)
ranked_drop: dict[int, float] = {}
for removed in (1, 2, 3):
    mask = torch.ones_like(hidden)
    mask[ranking[:removed]] = 0.0
    logits = clean.fork(f"remove_{removed}").do(tl.func("relu"), keep(mask)).output_ops[-1].out
    ranked_drop[removed] = float(clean_logits[0, target] - logits.detach()[0, target])

print(ranked_drop)
assert ranked_drop[3] >= ranked_drop[1]
RESULT = ranked_drop[3] >= ranked_drop[1]

Locality asks whether an intervention leaves unrelated outputs alone. Zero only the two features the target class reads (units 0 and 1): the target logit should fall and the other classes' logits should not move.

In [ ]:
# recipe-test: tool=torchlens expected=True
local = clean.fork("zero_target_evidence").do(
    tl.func("relu"), keep(torch.tensor([0.0, 0.0, 1.0, 1.0, 1.0]))
)
local_logits = local.output_ops[-1].out.detach()
target_drop = float(clean_logits[0, target] - local_logits[0, target])
others_moved = float((clean_logits[0, 1:] - local_logits[0, 1:]).abs().max())
print(f"target logit drop {target_drop:.2f}; largest move in the other classes {others_moved:.2e}")
assert target_drop > 0
assert torch.allclose(local_logits[0, 1:], clean_logits[0, 1:])
RESULT = target_drop > 0 and torch.allclose(local_logits[0, 1:], clean_logits[0, 1:])